In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from pathlib import Path
from collections import defaultdict
import torch
from torch.nn import functional as F
from safetensors.torch import load_file

from model import Qwen3ForCausalLM
from config import ModelConfig
from utils import load_from_safetensors

from transformers import AutoTokenizer

In [ ]:
model_path = Path.home() / "huggingface" / "Qwen3-0.6B"

In [ ]:
qwen3_state_dict = load_file(str(model_path / "model.safetensors"))

In [ ]:
config = ModelConfig(model_name="Qwen/Qwen3-0.6B", config_path=Path("models/qwen3/config.json"))

In [ ]:
dtype_map = {"bfloat16": torch.bfloat16, "float16": torch.float16, "float32": torch.float32}

In [ ]:
model = Qwen3ForCausalLM(config)

In [ ]:
load_from_safetensors(model, model_path)

In [ ]:
model.to(device="cuda", dtype=dtype_map[config.torch_dtype])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_path)

@torch.no_grad()
def generate(
    model: Qwen3ForCausalLM,
    tokenizer,
    prompt: str,
    max_new_tokens: int = 50,
    temperature: float = 1.0,
    top_k: int = None,
    eos_token_id: int = None,
    device: str = "cuda",
) -> str:
    """
    Simple autoregressive text generation.
    
    Args:
        model: The Qwen3ForCausalLM model
        tokenizer: HuggingFace tokenizer
        prompt: Input prompt string
        max_new_tokens: Maximum number of tokens to generate
        temperature: Sampling temperature (1.0 = no change, <1.0 = more deterministic)
        top_k: If set, only sample from top k tokens
        eos_token_id: Stop generation when this token is produced
        device: Device to run on
    
    Returns:
        Generated text string
    """
    model.eval()
    
    model.to(device)
    
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    
    if eos_token_id is None:
        eos_token_id = tokenizer.eos_token_id
    
    generated_ids = input_ids.clone()
    
    for _ in range(max_new_tokens):
        logits = model(generated_ids)
        
        next_token_logits = logits[:, -1, :]
        
        if temperature != 1.0:
            next_token_logits = next_token_logits / temperature
        
        if top_k is not None:
            top_k_vals, _ = torch.topk(next_token_logits, top_k, dim=-1)
            min_top_k = top_k_vals[:, -1].unsqueeze(-1)
            next_token_logits = torch.where(
                next_token_logits < min_top_k,
                torch.full_like(next_token_logits, float('-inf')),
                next_token_logits
            )
        
        probs = F.softmax(next_token_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        
        generated_ids = torch.cat([generated_ids, next_token], dim=-1)
        
        if next_token.item() == eos_token_id:
            break
    
    return tokenizer.decode(generated_ids[0], skip_special_tokens=True)

In [ ]:
output = generate(
    model,
    tokenizer,
    prompt="The capital of France is",
    max_new_tokens=30,
    temperature=0.7,
    top_k=50,
)
print(output)